<a href="https://colab.research.google.com/github/grantfahey/DSAC-Project/blob/main/DSAC_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import pandas as pd

seasons = [2023, 2024, 2025]
pbp = pd.concat([
    pd.read_csv(f"https://github.com/nflverse/nflverse-data/releases/download/pbp/play_by_play_{s}.csv.gz", low_memory=False)
    for s in seasons
])

fourth = pbp[pbp["down"] == 4]
print(len(fourth), "fourth down plays")

13059 fourth down plays


In [14]:
go = fourth[fourth["play_type"].isin(["run", "pass"])]

rates = go[go["ydstogo"] <= 10].groupby("ydstogo")["fourth_down_converted"].agg(["mean", "count"])
rates["mean"] = (rates["mean"] * 100).round(1)
rates.columns = ["conversion %", "attempts"]
rates

,conversion %,attempts
ydstogo,,
1,69.4,1010
2,59.5,390
3,54.1,281
4,52.0,198
5,50.3,157
6,34.4,93
7,34.8,69
8,27.5,69
9,25.0,28


In [15]:
fg = fourth[fourth["play_type"] == "field_goal"].copy()
fg["made"] = fg["field_goal_result"] == "made"
fg["distance"] = (fg["kick_distance"] // 5) * 5

fg_rates = fg.groupby("distance")["made"].agg(["mean", "count"])
fg_rates["mean"] = (fg_rates["mean"] * 100).round(1)
fg_rates.columns = ["make %", "attempts"]
fg_rates

,make %,attempts
distance,,
15.0,100.0,5
20.0,99.3,267
25.0,97.1,421
30.0,97.3,440
35.0,89.9,437
40.0,83.7,442
45.0,75.8,405
50.0,74.2,476
55.0,65.9,208


In [16]:
punts = fourth[fourth["play_type"] == "punt"].copy()
punts["return_yards"] = punts["return_yards"].fillna(0)
punts["opp_start"] = punts["yardline_100"] - punts["kick_distance"] + punts["return_yards"]
punts.loc[punts["touchback"] == 1, "opp_start"] = 20
punts["spot"] = (punts["yardline_100"] // 10) * 10

punt_results = punts.groupby("spot")["opp_start"].agg(["mean", "count"]).round(1)
punt_results.columns = ["opp starts at their own", "punts"]
punt_results

,opp starts at their own,punts
spot,,
20.0,20.5,2
30.0,10.6,63
40.0,12.0,802
50.0,14.9,1263
60.0,20.9,1773
70.0,30.2,1559
80.0,40.2,794
90.0,50.3,257
